In [1]:
# type: ignore


import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


load_dotenv()

llm_model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME"),
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),
    extra_body={"enable_thinking": False},
)

In [2]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class RouteQuery(BaseModel):
    """Route a user query to the most relevant destination."""

    datasource: Literal["vector_store", "web_search", "direct_answer"] = Field(
        ...,
        description=(
            "Choose 'vector_store' for questions about Agentic RAG or RAG architectures; "
            "'web_search' for questions requiring up-to-date or external information; "
            "'direct_answer' for greetings, simple factual questions (e.g., capital of China), "
            "or anything the LLM can confidently answer without retrieval."
        ),
    )


structured_llm_router = llm_model.with_structured_output(RouteQuery)

system_prompt = system_prompt = """
You are an expert router that decides how to handle a user question.

- **Route to "vector_store"** if the question is about:
    • Agentic Retrieval-Augmented Generation (Agentic RAG)
    • Agent-based RAG systems
    • RAG workflow patterns (multi-agent, adaptive, self-correction, etc.)

- **Route to "web_search"** if the question:
    • Requires current information (e.g., stock prices, news, sports results)
    • Is about topics outside your training data or too niche
    • Cannot be answered confidently from general knowledge alone

- **Route to "direct_answer"** if the question is:
    • A greeting (e.g., "Hi", "Hello", "How are you?")
    • A simple factual question with a well-known answer (e.g., "What is the capital of China?", "Who wrote Hamlet?")
    • A basic conceptual question that doesn't need retrieval (e.g., "What is photosynthesis?")
    • Any query the LLM can answer directly and accurately from its internal knowledge

You MUST output a valid JSON object with ONLY the key "datasource"."""

route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

question_router = route_prompt | structured_llm_router

In [3]:
question_router.invoke({"question": "什么是 Agentic RAG？"})

RouteQuery(datasource='vector_store')

In [4]:
question_router.invoke({"question": "你好"})

RouteQuery(datasource='direct_answer')

In [6]:
question_router.invoke({"question": "明天天气怎么样"})

RouteQuery(datasource='web_search')